# 03 · Busca por palavra-chave x busca semântica

Agora a parte divertida: **descobrir quem disse** a partir de uma ideia, e não da frase exata.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore", category=UserWarning)   # aviso do pandas sobre conexão pyodbc
sys.path.append(str(pathlib.Path.cwd().parent))   # permite "from src import ..."
from src import config, banco
import time
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from src.vetores import IndiceEmMemoria, vetor_para_bytes

conn = banco.conectar(config.SQL_DATABASE)
modelo = SentenceTransformer(config.MODELO_EMBEDDING)
indice = IndiceEmMemoria.do_banco(conn, config.MODELO_EMBEDDING)
frases = pd.read_sql("SELECT * FROM rag.vw_Frases", conn).set_index("FraseId")
print("Índice em memória:", indice.matriz.shape, f"({indice.matriz.nbytes/1024:.0f} KB)")

c:\dev\Estudo-MDS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1467.13it/s]


Índice em memória: (31, 384) (46 KB)


## 1. Busca por palavra-chave (LIKE)

In [2]:
def busca_palavra(termo):
    return pd.read_sql("EXEC rag.usp_BuscaPalavraChave @Termo = ?", conn, params=[termo])

busca_palavra("medo")

,FraseId,Politico,Texto,Ano
0,15,Franklin D. Roosevelt,A única coisa que devemos temer é o próprio medo.,1933


In [3]:
busca_palavra("coragem")   # a ideia existe no banco, mas a palavra não

,FraseId,Politico,Texto,Ano


## 2. Busca semântica (cálculo no Python)

In [4]:
def quem_disse(pergunta, k=3):
    vq = modelo.encode(pergunta, normalize_embeddings=True)
    res = indice.buscar(vq, k)
    out = frases.loc[[fid for fid, _ in res], ["Politico", "Texto", "Ano"]].copy()
    out.insert(0, "Similaridade", [round(s, 3) for _, s in res])
    return out

quem_disse("coragem para enfrentar o pânico")

,Similaridade,Politico,Texto,Ano
FraseId,,,,
15,0.569,Franklin D. Roosevelt,A única coisa que devemos temer é o próprio medo.,1933
3,0.290,Tancredo Neves,"Não vamos nos dispersar. Continuemos reunidos,...",1985
8,0.288,Dom Pedro I,Independência ou morte!,1822


## 3. Lado a lado: palavra-chave x significado

In [5]:
perguntas = [
    "não desistir nunca da luta",
    "servir a pátria em vez de pedir coisas ao governo",
    "democracia é o povo governando",
    "cansado de ver a corrupção vencer",
    "derrubar a barreira que divide a cidade",
    "um país com dificuldade de manter a união",
]
for p in perguntas:
    kw = busca_palavra(p)
    top = quem_disse(p, 1).iloc[0]
    print(f"\n{p}")
    print(f"   LIKE: {len(kw)} resultado(s)")
    print(f"   Semântica: {top.Politico} ({top.Similaridade}) → {top.Texto[:70]}")


não desistir nunca da luta
   LIKE: 0 resultado(s)
   Semântica: Winston Churchill (0.621) → Lutaremos nas praias, lutaremos nos campos e nas ruas, lutaremos nas c

servir a pátria em vez de pedir coisas ao governo
   LIKE: 0 resultado(s)
   Semântica: John F. Kennedy (0.574) → Não pergunte o que o seu país pode fazer por você; pergunte o que você

democracia é o povo governando
   LIKE: 0 resultado(s)
   Semântica: Abraham Lincoln (0.564) → O governo do povo, pelo povo e para o povo não desaparecerá da Terra.

cansado de ver a corrupção vencer
   LIKE: 0 resultado(s)
   Semântica: Gilmar Mendes (0.577) → Até a máfia tem ética. É preciso ter decência.

derrubar a barreira que divide a cidade
   LIKE: 0 resultado(s)
   Semântica: Abraham Lincoln (0.371) → Uma casa dividida contra si mesma não se sustenta.

um país com dificuldade de manter a união
   LIKE: 0 resultado(s)
   Semântica: Luiz Inácio Lula da Silva (0.514) → Nunca antes na história deste país.


## 4. Busca entre idiomas
O modelo é multilíngue: perguntar em inglês ou espanhol encontra a frase gravada em português.

In [12]:
quem_disse("traficante ", 2)

,Similaridade,Politico,Texto,Ano
FraseId,,,,
23,0.423,Jair Bolsonaro,"Não sou coveiro, tá?",2020
27,0.348,Luís Roberto Barroso,"Perdeu, mané, não amola.",2022


In [7]:
quem_disse("tenemos que derribar el muro", 2)

,Similaridade,Politico,Texto,Ano
FraseId,,,,
16,0.628,Ronald Reagan,"Senhor Gorbachev, derrube este muro!",1987
21,0.533,Angela Merkel,Nós vamos conseguir.,2015


## 5. O "R" do RAG: montando o contexto para um LLM

RAG = **R**etrieval (buscar) + **A**ugmented (enriquecer o prompt) + **G**eneration (o LLM responde). O SQL Server 2022 + Python resolvem o **R**. O prompt abaixo pode ser enviado para qualquer LLM.

In [13]:
def montar_prompt(pergunta, k=3):
    res = quem_disse(pergunta, k)
    contexto = "\n".join(f"- \"{r.Texto}\" ({r.Politico}, {r.Ano})" for r in res.itertuples())
    return f"""Responda usando APENAS as frases abaixo. Se não houver resposta, diga que não sabe.

Frases recuperadas do SQL Server:
{contexto}

Pergunta: {pergunta}"""

print(montar_prompt("Quem falou sobre não ter medo?"))

Responda usando APENAS as frases abaixo. Se não houver resposta, diga que não sabe.

Frases recuperadas do SQL Server:
- "A única coisa que devemos temer é o próprio medo." (Franklin D. Roosevelt, 1933)
- "Não sou coveiro, tá?" (Jair Bolsonaro, 2020)
- "Perdeu, mané, não amola." (Luís Roberto Barroso, 2022)

Pergunta: Quem falou sobre não ter medo?


## 6. Bônus para DBAs: cosseno em T-SQL puro no 2022

Sem `VECTOR` e sem `VECTOR_DISTANCE` (só no 2025), mas com a função `rag.fn_VetorParaLinhas`, que usa o `GENERATE_SERIES` do 2022 para ler os bytes do `VARBINARY`, e um `SUM` de produtos. A pergunta vai para o banco no **mesmo formato** dos vetores gravados. Funciona, e é ótimo para entender a matemática. Na prática, para volume, a aplicação é mais rápida.

In [14]:
pergunta = "coragem para enfrentar o pânico"
vq = modelo.encode(pergunta, normalize_embeddings=True)

t = time.perf_counter()
tsql = pd.read_sql("EXEC rag.usp_BuscaSemanticaTSQL @Vetor = ?, @Modelo = ?, @K = 3",
                   conn, params=[vetor_para_bytes(vq), config.MODELO_EMBEDDING])
print(f"T-SQL: {(time.perf_counter()-t)*1000:.1f} ms")
tsql

T-SQL: 586.9 ms


,FraseId,Politico,Texto,Similaridade
0,15,Franklin D. Roosevelt,A única coisa que devemos temer é o próprio medo.,0.569373
1,3,Tancredo Neves,"Não vamos nos dispersar. Continuemos reunidos,...",0.289977
2,8,Dom Pedro I,Independência ou morte!,0.288165


O resultado bate com o do Python? Deveria: é a mesma conta, feita em lugares diferentes.

In [15]:
quem_disse(pergunta, 3)

,Similaridade,Politico,Texto,Ano
FraseId,,,,
15,0.569,Franklin D. Roosevelt,A única coisa que devemos temer é o próprio medo.,1933
3,0.290,Tancredo Neves,"Não vamos nos dispersar. Continuemos reunidos,...",1985
8,0.288,Dom Pedro I,Independência ou morte!,1822


## 7. E se forem 1 milhão de frases?

Simulação com vetores aleatórios para medir a força bruta em NumPy.

In [16]:
n, d = 1_000_000, 384
mat = np.random.randn(n, d).astype("float32")
mat /= np.linalg.norm(mat, axis=1, keepdims=True)
q = mat[0]
t = time.perf_counter()
top = np.argsort(-(mat @ q))[:5]
print(f"{n:,} vetores | {mat.nbytes/1024**3:.2f} GB em RAM | busca em {(time.perf_counter()-t)*1000:.0f} ms")
del mat

1,000,000 vetores | 1.43 GB em RAM | busca em 196 ms


**Leitura do resultado:**
- Até centenas de milhares de vetores, força bruta em memória resolve.
- Milhões de vetores: use um índice ANN em memória (FAISS, hnswlib) carregado a partir do SQL Server.
- O SQL Server continua sendo a **fonte da verdade**: backup, segurança, transação e governança.
- Quando migrar para o 2025, os bytes viram `VECTOR(384)` e a conta vai para `VECTOR_DISTANCE`.